# 01 — Corpus EDA

Inventory of the Atlas Industries corpus: file formats, languages, sizes, and
a spot-check that Arabic text extracts correctly (FR-B3).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.chunking import detect_language
from src.loaders import SUPPORTED_EXTENSIONS, load_document
from src.settings import settings

print("docs dir:", settings.docs_dir)

## Format × language inventory

In [ ]:
rows = []
for domain in settings.domains:
    for path in sorted((settings.docs_dir / domain).iterdir()):
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        text = load_document(path)
        rows.append(
            {
                "domain": domain,
                "file": path.name,
                "format": path.suffix.lower().lstrip("."),
                "language": detect_language(text),
                "chars": len(text),
            }
        )

import pandas as pd

df = pd.DataFrame(rows)
df

In [ ]:
print("Documents per domain:")
print(df["domain"].value_counts().to_string())
print("\nDocuments per format (brief expects ~4 md / 3 pdf / 3 docx per domain):")
print(df.groupby(["domain", "format"]).size().to_string())
print("\nDocuments per language (brief expects ~60% en / 30% ar / 10% bilingual):")
print(df["language"].value_counts(normalize=True).round(2).to_string())

## FR-B3 — Arabic extraction sanity check

Print raw excerpts from one Arabic PDF and one Arabic DOCX. Scrambled glyphs or
character-reversed lines mean the PDF loader is wrong (see `src/loaders.py` for
why `pypdf` was chosen over `pdfplumber`).

In [ ]:
arabic_samples = {
    "PDF": settings.docs_dir / "finance" / "FIN-008-بدل-المواصلات.pdf",
    "DOCX": settings.docs_dir / "finance" / "FIN-009-عشاء-العملاء.docx",
    "MD": settings.docs_dir / "hr" / "HR-007-سياسة-الإجازات-المرضية.md",
}

for kind, path in arabic_samples.items():
    text = load_document(path)
    print(f"=== {kind}: {path.name} ({len(text)} chars) ===")
    print(text[:400].replace("\n", " | "))
    print()

## Chunking profile

How many chunks does the current `CHUNK_SIZE` / `CHUNK_OVERLAP` produce?
On a 30-doc corpus this drives retrieval ranking sensitivity (known concern).

In [ ]:
from src.ingest import build_chunks

chunks = build_chunks()
print(f"{len(chunks)} chunks from {len(rows)} documents")
print("by domain:", pd.Series([c.domain for c in chunks]).value_counts().to_dict())
print("by language:", pd.Series([c.language for c in chunks]).value_counts().to_dict())
print("avg chunk chars:", int(pd.Series([len(c.text) for c in chunks]).mean()))
print("\nArabic filenames preserved through chunking:")
for c in chunks:
    if any("\u0600" <= ch <= "\u06ff" for ch in c.source_filename):
        print(" ", c.chunk_id, "->", c.source_filename)
        break